# Cross-Sectional Stock Return Prediction

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
#Chosen stocks
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "JPM", "XOM", "JNJ", "PG"
]

In [4]:
#Download data
prices = yf.download(tickers, start='2015-01-01', end = '2026-01-01', auto_adjust=True)
prices.head()

[*********************100%***********************]  10 of 10 completed


Price           Close                                                       \
Ticker           AAPL     AMZN      GOOGL        JNJ        JPM       META   
Date                                                                         
2015-01-02  24.171751  15.4260  26.227924  75.737381  45.838112  77.705772   
2015-01-05  23.490803  15.1095  25.728178  75.208389  44.415066  76.457703   
2015-01-06  23.493013  14.7645  25.093222  74.838837  43.263432  75.427574   
2015-01-07  23.822433  14.9210  25.019423  76.490997  43.329433  75.427574   
2015-01-08  24.737751  15.0230  25.106594  77.092407  44.297699  77.438316   

Price                                                  ...     Volume  \
Ticker           MSFT      NVDA         PG        XOM  ...       AAPL   
Date                                                   ...              
2015-01-02  39.607178  0.481884  64.642082  56.777920  ...  212818400   
2015-01-05  39.242970  0.473745  64.334709  55.224335  ...  257142000   
2015-01-06  38.666985  0.459382  64.041672  54.930782  ...  263188400   
2015-01-07  39.158249  0.458185  64.377617  55.487358  ...  160423600   
2015-01-08  40.310211  0.475420  65.113792  56.410923  ...  237458000   

Price                                                                  \
Ticker          AMZN     GOOGL      JNJ       JPM      META      MSFT   
Date                                                                    
2015-01-02  55664000  26480000  5753600  12600000  18177500  27913900   
2015-01-05  55484000  41182000  8079300  20100600  26452200  39673900   
2015-01-06  70380000  54456000  7428000  29074100  27399300  36447900   
2015-01-07  52806000  46918000  7931700  23843200  22045300  29114100   
2015-01-08  61768000  73054000  9916000  16971100  23961000  29645200   

Price                                     
Ticker           NVDA       PG       XOM  
Date                                      
2015-01-02  113680000  7251400  10220400  
2015-01-05  197952000  8626100  18502400  
2015-01-06  197764000  7791200  16670700  
2015-01-07  321808000  5986600  13590700  
2015-01-08  283780000  6823300  15487500  

[5 rows x 50 columns]

In [5]:
close = prices["Close"] #Closing prices (daily)
prices_monthly = close.resample("ME").last() #Month's end closing prices
returns_monthly = prices_monthly.pct_change() #Calculate returns

returns_monthly.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
Date,,,,,,,,,,
2015-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-02-28,0.100776,0.072293,0.046656,0.030833,0.126885,0.040311,0.093120,0.153382,0.009965,0.020435
2015-03-31,-0.031372,-0.021202,-0.014094,-0.018633,-0.011423,0.041155,-0.072748,-0.051225,-0.037472,-0.039982
2015-04-30,0.005786,0.133513,-0.010691,-0.013916,0.051180,-0.041961,0.196262,0.060679,-0.021851,0.027882
2015-05-31,0.045339,0.017663,-0.006287,0.016837,0.039835,0.005332,-0.030334,0.001461,-0.014086,-0.016771


In [6]:
#target variable
future_returns = returns_monthly.shift(-1)
future_returns.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
Date,,,,,,,,,,
2015-01-31,0.100776,0.072293,0.046656,0.030833,0.126885,0.040311,0.093120,0.153382,0.009965,0.020435
2015-02-28,-0.031372,-0.021202,-0.014094,-0.018633,-0.011423,0.041155,-0.072748,-0.051225,-0.037472,-0.039982
2015-03-31,0.005786,0.133513,-0.010691,-0.013916,0.051180,-0.041961,0.196262,0.060679,-0.021851,0.027882
2015-04-30,0.045339,0.017663,-0.006287,0.016837,0.039835,0.005332,-0.030334,0.001461,-0.014086,-0.016771
2015-05-31,-0.037228,0.011323,-0.009682,-0.026763,0.030101,0.083091,-0.057832,-0.091279,-0.001913,-0.023474


## Feature Engineering

In [7]:
# features_df = future_returns.reset_index().melt(
#     id_vars="Date",
#     var_name="Ticker",
#     value_name="Future Returns"
# )

# features_df.head()


### Momentum

The momentum at time $t$ is given by $$ \frac{P_{t-1}}{P_{t-12}} - 1 $$

measuring how much the stock has increased or decreaed over the past year

In [8]:
momentum = prices_monthly.shift(1) / prices_monthly.shift(12) - 1

In [9]:
momentum_df = momentum.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Momentum"
)

features_df = momentum_df.copy()
features_df.tail()

,Date,Ticker,Momentum
1315,2025-08-31,XOM,-0.028234
1316,2025-09-30,XOM,0.010258
1317,2025-10-31,XOM,0.000400
1318,2025-11-30,XOM,-0.003652
1319,2025-12-31,XOM,0.117174


### Volatility

In [10]:
volatility = prices_monthly.rolling(12).std()

In [11]:
volatility_df = volatility.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Volatility"
)
features_df['Volatility'] = volatility_df['Volatility']
features_df.tail()

,Date,Ticker,Momentum,Volatility
1315,2025-08-31,XOM,-0.028234,4.983384
1316,2025-09-30,XOM,0.010258,4.959668
1317,2025-10-31,XOM,0.000400,5.077393
1318,2025-11-30,XOM,-0.003652,5.302773
1319,2025-12-31,XOM,0.117174,5.991955


### Beta

In [12]:
#Download market data
market = yf.download(
    "^GSPC",
    start="2015-01-01",
    end="2026-01-01",
    auto_adjust=True
)

market_prices = market["Close"].resample("ME").last()
market_returns = market_prices.squeeze().pct_change()

[*********************100%***********************]  1 of 1 completed


In [13]:
returns_monthly = returns_monthly.astype(float)
market_returns = market_returns.astype(float)


In [14]:
#Calculate betas
betas = {}
for ticker in tickers:
    betas[ticker] = returns_monthly[ticker].rolling(12).cov(market_returns) / market_returns.rolling(12).var()

betas

{'AAPL': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    0.368903
 2025-09-30    0.493241
 2025-10-31    0.463810
 2025-11-30    0.389568
 2025-12-31    0.619201
 Freq: ME, Length: 132, dtype: float64,
 'MSFT': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    1.342136
 2025-09-30    1.298323
 2025-10-31    1.202574
 2025-11-30    1.352794
 2025-12-31    1.473589
 Freq: ME, Length: 132, dtype: float64,
 'GOOGL': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    0.879256
 2025-09-30    1.034016
 2025-10-31    1.122436
 2025-11-30    1.435124
 2025-12-31    1.908739
 Freq: ME, Length: 132, dtype: float64,
 'AMZN': Date
 2015-01-31         NaN
 2015-0

In [15]:
betas_predf = pd.DataFrame(betas)
betas_df = betas_predf.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Beta"
)

betas_df

,Date,Ticker,Beta
0,2015-01-31,AAPL,NaN
1,2015-02-28,AAPL,NaN
2,2015-03-31,AAPL,NaN
3,2015-04-30,AAPL,NaN
4,2015-05-31,AAPL,NaN
...,...,...,...
1315,2025-08-31,PG,0.552709
1316,2025-09-30,PG,0.501826
1317,2025-10-31,PG,0.454634
1318,2025-11-30,PG,0.190867


In [16]:
features_df['Beta'] = betas_df['Beta']
features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta
1315,2025-08-31,XOM,-0.028234,4.983384,0.552709
1316,2025-09-30,XOM,0.010258,4.959668,0.501826
1317,2025-10-31,XOM,0.000400,5.077393,0.454634
1318,2025-11-30,XOM,-0.003652,5.302773,0.190867
1319,2025-12-31,XOM,0.117174,5.991955,0.057402


### Current Size \& Value

Note: need to be adjusted into historical data for backtesting, but temporarily uses current data

In [17]:
market_caps = {}
pe_ratios = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    market_caps[ticker] = info.get('marketCap')
    pe_ratios[ticker] = info.get('trailingPE')

In [18]:
market_caps

{'AAPL': 5074761285632,
 'MSFT': 3894773022720,
 'GOOGL': 3916147458048,
 'AMZN': 2675005718528,
 'META': 1637078401024,
 'NVDA': 5094050889728,
 'JPM': 880543858688,
 'XOM': 753343397888,
 'JNJ': 635072544768,
 'PG': 373209432064}

In [19]:
#Use log market cap instead for feature
log_market_caps = {
    ticker : float(np.log(cap))
    for ticker, cap in market_caps.items()
    if cap is not None
}

log_market_caps

{'AAPL': 29.25530060243622,
 'MSFT': 28.990656519540426,
 'GOOGL': 28.9961294951606,
 'AMZN': 28.61497263404171,
 'META': 28.123934306282496,
 'NVDA': 29.259094482567047,
 'JPM': 27.50380557475855,
 'XOM': 27.347787000483706,
 'JNJ': 27.17700507304292,
 'PG': 26.64540557908362}

In [20]:
#P/E ratios to Earnings yield

earnings_yield = {
    ticker : 1 / pe
    for ticker, pe in pe_ratios.items()
    if pe is not None and pe > 0
}

earnings_yield

{'AAPL': 0.0250772882022393,
 'MSFT': 0.03422241806571334,
 'GOOGL': 0.06224041080662826,
 'AMZN': 0.050120969465854315,
 'META': 0.04561015902438045,
 'NVDA': 0.03749525685000847,
 'JPM': 0.07189818986241274,
 'XOM': 0.04241034802313281,
 'JNJ': 0.024980100851661566,
 'PG': 0.041237110001063094}

In [21]:
curr_df = pd.DataFrame({
    "Ticker" : list(log_market_caps.keys()),
    "Size" : log_market_caps.values(),
    "Value" : earnings_yield.values()
})

curr_df.head()

,Ticker,Size,Value
0,AAPL,29.255301,0.025077
1,MSFT,28.990657,0.034222
2,GOOGL,28.996129,0.062240
3,AMZN,28.614973,0.050121
4,META,28.123934,0.045610


In [22]:
features_df = features_df.merge(
    curr_df,
    on="Ticker",
    how="left"
)

features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta,Size,Value
1315,2025-08-31,XOM,-0.028234,4.983384,0.552709,27.347787,0.04241
1316,2025-09-30,XOM,0.010258,4.959668,0.501826,27.347787,0.04241
1317,2025-10-31,XOM,0.000400,5.077393,0.454634,27.347787,0.04241
1318,2025-11-30,XOM,-0.003652,5.302773,0.190867,27.347787,0.04241
1319,2025-12-31,XOM,0.117174,5.991955,0.057402,27.347787,0.04241


In [23]:
target_df = future_returns.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Future Returns"
)

features_df['Future Returns'] = target_df['Future Returns']
features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta,Size,Value,Future Returns
1315,2025-08-31,XOM,-0.028234,4.983384,0.552709,27.347787,0.04241,-0.013474
1316,2025-09-30,XOM,0.010258,4.959668,0.501826,27.347787,0.04241,0.014279
1317,2025-10-31,XOM,0.000400,5.077393,0.454634,27.347787,0.04241,0.022507
1318,2025-11-30,XOM,-0.003652,5.302773,0.190867,27.347787,0.04241,0.038130
1319,2025-12-31,XOM,0.117174,5.991955,0.057402,27.347787,0.04241,NaN


## Data Cleaning

remove NaN values

In [24]:
data = features_df.dropna(subset=['Momentum', 'Volatility', 'Beta', 'Size', 'Value', 'Future Returns']).copy()

print(data.shape)
print(data.head())

(1190, 8)
         Date Ticker  Momentum  Volatility      Beta       Size     Value  \
12 2016-01-31   AAPL -0.086263    2.129119  1.279007  29.255301  0.025077   
13 2016-02-29   AAPL -0.232373    2.380288  1.116989  29.255301  0.025077   
14 2016-03-31   AAPL -0.208532    2.333380  1.340057  29.255301  0.025077   
15 2016-04-30   AAPL -0.112981    2.539116  1.320436  29.255301  0.025077   
16 2016-05-31   AAPL -0.270183    2.278699  1.347971  29.255301  0.025077   

    Future Returns  
12       -0.001288  
13        0.127211  
14       -0.139921  
15        0.071773  
16       -0.042660  


split data

In [25]:
data['Date'] = pd.to_datetime(data['Date'])

train = data[data['Date'] < '2024-01-01'].copy()
test = data[data['Date'] >= '2024-01-01'].copy()

print(len(train) , len(test))

960 230


In [26]:
features = ['Momentum', 'Volatility', 'Beta', 'Size', 'Value']

X_train = train[features]
y_train = train['Future Returns']

X_test = test[features]
y_test = test['Future Returns']

## Baseline Linear Regression Model

In [27]:
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

preds_lm = lm.predict(X_test)
test['Predicted Returns'] = preds_lm

In [28]:
from sklearn.metrics import mean_squared_error, r2_score

rmse_lm = np.sqrt(mean_squared_error(y_test, preds_lm))
r2_lm = r2_score(y_test, preds_lm)

print('RMSE:', rmse_lm)
print('R2:', r2_lm)

RMSE: 0.07168404879800926
R2: 0.010323602008825872


In [29]:
monthly_ic = test.groupby('Date').apply(
    lambda x: x['Predicted Returns'].corr(x['Future Returns']),
    include_groups=False
)

print("LM Mean Monthly IC:", monthly_ic.mean())
print("LM Monthly IC St. Dev.:", monthly_ic.std())

LM Mean Monthly IC: 0.14372025056010782
LM Monthly IC St. Dev.: 0.3874418527741401


## Ridge Regression

In [30]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

val_years = [2020, 2021, 2022, 2023]
alphas = [0.01, 0.1, 1.0, 10, 100, 1000, 10000, 100000]

results = []

for alpha in alphas:
    rmses = []
    for year in val_years:
        train_fold = data[data['Date'].dt.year < year]
        val_fold = data[data['Date'].dt.year == year]

        ridge = make_pipeline(
            StandardScaler(),
            Ridge(alpha=alpha)
        )

        ridge.fit(train_fold[features], train_fold['Future Returns'])
        preds_ridge = ridge.predict(val_fold[features])

        rmse = np.sqrt(mean_squared_error(val_fold['Future Returns'], preds_ridge))
        rmses.append(rmse)

    results.append({
        "alpha": alpha,
        "mean_val_rmse": np.mean(rmses)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values("mean_val_rmse"))
    

       alpha  mean_val_rmse
5    1000.00       0.094715
6   10000.00       0.094865
7  100000.00       0.094922
4     100.00       0.094934
3      10.00       0.095057
2       1.00       0.095073
1       0.10       0.095074
0       0.01       0.095074


In [31]:
ridge = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1000)
    )

ridge.fit(X_train, y_train)
preds_ridge = ridge.predict(X_test)
test['Predicted Returns_Ridge'] = preds_ridge

In [32]:
rmse_ridge = np.sqrt(mean_squared_error(y_test, preds_ridge))
r2_ridge = r2_score(y_test, preds_ridge)

print('RMSE:', rmse_ridge)
print('R2:', r2_ridge)

RMSE: 0.07158053290098514
R2: 0.013179837725398436


There is a slightly lower RMSE (-0.0002) and slightly higher R2 (+0.0034) in the Ridge regression model than the baseline linear model.

In [33]:
mic_ridge = test.groupby('Date').apply(
    lambda x: x['Predicted Returns_Ridge'].corr(x['Future Returns']),
    include_groups=False
)

print("Ridge Mean Monthly IC:", mic_ridge.mean())
print("Ridge Monthly IC St. Dev.:", mic_ridge.std())

Ridge Mean Monthly IC: 0.14840185689600527
Ridge Monthly IC St. Dev.: 0.39334672295624745


## XGBoost Regressor

In [49]:
from xgboost import XGBRegressor
from itertools import product

val_years = [2020, 2021, 2022, 2023]
trees = [100, 200, 300, 400, 500]
depths =[2, 3, 4]
lrs = [0.005, 0.01, 0.05, 0.1]

params = list(product(
    trees,
    depths,
    lrs
))

results_xgb = []

for num_trees, depth, lr in params:
    rmses = []
    for year in val_years:
        train_fold = data[data['Date'].dt.year < year]
        val_fold = data[data['Date'].dt.year == year]

        if train_fold.empty or val_fold.empty:
            continue

        xgb = XGBRegressor(
            n_estimators=num_trees,
            max_depth=depth,
            learning_rate=lr,
            objective='reg:squarederror',
            random_state=42
        )

        xgb.fit(train_fold[features], train_fold['Future Returns'])
        preds_xgb = xgb.predict(val_fold[features])

        rmse = np.sqrt(mean_squared_error(val_fold['Future Returns'], preds_xgb))
        rmses.append(rmse)

    results_xgb.append({
        "num_trees": num_trees,
        "depth": depth,
        "learnng_rate": lr,
        "mean_val_rmse": np.mean(rmses)
    })

results_df = pd.DataFrame(results_xgb)
print(results_df.sort_values("mean_val_rmse").head(10))

    num_trees  depth  learnng_rate  mean_val_rmse
0         100      2         0.005       0.095028
12        200      2         0.005       0.095091
1         100      2         0.010       0.095091
24        300      2         0.005       0.095173
4         100      3         0.005       0.095234
36        400      2         0.005       0.095273
13        200      2         0.010       0.095276
5         100      3         0.010       0.095384
16        200      3         0.005       0.095389
48        500      2         0.005       0.095479


In [ ]:
xgb = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.01,
    objective='reg:squarederror',
    random_state=42
    )

xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_test)
test['Predicted Returns_XGB'] = preds_xgb

In [58]:
rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
r2_xgb = r2_score(y_test, preds_xgb)

print('RMSE:', rmse_xgb)
print('R2:', r2_xgb)

RMSE: 0.07215038842331735
R2: -0.0025949357075707447
